# Notebook 04 — Simulation Study: ColdGuard vs MKT vs VVM

Runs 1,000 synthetic India cold chain scenarios and compares three decision protocols:
- **ColdGuard** (kinetic-Bayesian, this project)
- **MKT** (WHO Mean Kinetic Temperature threshold)
- **VVM** (proxy: discard if cumulative heat units exceed VVM calibration)

Metrics: False Discard Rate (FDR), False Use Rate (FUR), Accuracy, McNemar's test.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

from simulation.cold_chain_generator import generate_scenario, INDIA_DISTRICT, INDIA_PHC, INDIA_OUTREACH
from core.utils import run_analysis
from core.decision import Decision
from core.arrhenius import compute_mkt, integrate_degradation
from core.vaccine_params import VACCINE_DB

rng = np.random.default_rng(2024)
N_SCENARIOS = 1000
VACCINE_TYPE = 'DPT'
MC_SAMPLES = 500  # lower for notebook speed; use 5000 for paper

## 1. Generate Scenarios and Ground Truth

Ground truth: apply Arrhenius at exact (noise-free) Ea/A to determine true potency.
Label = DISCARD if true potency < min_potency_threshold, else USE.
(No INVESTIGATE in ground truth — that's a protocol-level uncertainty zone.)

In [ ]:
vparams = VACCINE_DB[VACCINE_TYPE]
records = []

profiles = [INDIA_DISTRICT, INDIA_PHC, INDIA_OUTREACH]
profile_names = ['District', 'PHC', 'Outreach']
profile_weights = [0.3, 0.4, 0.3]

for i in range(N_SCENARIOS):
    profile = rng.choice(profiles, p=profile_weights)
    ts, temps = generate_scenario(profile, rng=rng)

    # Ground truth (no uncertainty)
    from core.arrhenius import integrate_degradation, compute_potency
    D_true = integrate_degradation(ts, temps, vparams.Ea_mean, vparams.A)
    true_potency = np.exp(-D_true)
    ground_truth = Decision.USE if true_potency >= vparams.min_potency_threshold else Decision.DISCARD

    records.append({'ts': ts, 'temps': temps, 'true_potency': true_potency, 'ground_truth': ground_truth})

print(f"Generated {N_SCENARIOS} scenarios")
gt_counts = pd.Series([r['ground_truth'].value for r in records]).value_counts()
print(gt_counts)

## 2. Run All Three Protocols

In [ ]:
def mkt_decision(ts, temps, vaccine_type):
    """Decision via WHO MKT: if MKT potency < min_potency → DISCARD."""
    from core.arrhenius import compute_mkt, mkt_potency_estimate
    vp = VACCINE_DB[vaccine_type]
    mkt_pot = mkt_potency_estimate(ts, temps, vp)
    if mkt_pot >= vp.min_potency_threshold:
        return Decision.USE
    return Decision.DISCARD


def vvm_decision(ts, temps, vaccine_type):
    """VVM proxy: accumulate heat units using VVM4 calibration (T_ref=37°C).
       VVM4 endpoint ≈ 2 days at 37°C = 48 h·°C above 7°C equivalent.
       Simplified: discard if any period exceeds VVM-equivalent discard condition.
    """
    # Simple proxy: DISCARD if any temperature > 37°C OR if extended excursion > 8°C for >8h
    temps_arr = np.array(temps)
    dt_hours = np.diff(np.array(ts)) / 3600
    above_8 = temps_arr[:-1] > 8.0
    hours_above_8 = (dt_hours * above_8).sum()
    if np.any(temps_arr > 37.0) or hours_above_8 > 8:
        return Decision.DISCARD
    return Decision.USE


print("Running protocols on all scenarios (this may take a minute)...")
for i, rec in enumerate(records):
    result = run_analysis(VACCINE_TYPE, rec['ts'], rec['temps'], n_mc_samples=MC_SAMPLES)
    cg_dec = result['decision_output'].decision
    rec['coldguard'] = cg_dec if cg_dec != Decision.INVESTIGATE else Decision.DISCARD  # conservative
    rec['mkt'] = mkt_decision(rec['ts'], rec['temps'], VACCINE_TYPE)
    rec['vvm'] = vvm_decision(rec['ts'], rec['temps'], VACCINE_TYPE)
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{N_SCENARIOS}")

print("Done.")

## 3. Compute FDR / FUR / Accuracy

In [ ]:
def compute_metrics(records, protocol_key):
    tp = fp = tn = fn = 0
    for r in records:
        gt = r['ground_truth']
        pred = r[protocol_key]
        if gt == Decision.USE and pred == Decision.USE:
            tp += 1  # correctly kept
        elif gt == Decision.USE and pred == Decision.DISCARD:
            fp += 1  # false discard
        elif gt == Decision.DISCARD and pred == Decision.DISCARD:
            tn += 1  # correctly discarded
        else:
            fn += 1  # false use
    total = len(records)
    n_use_gt = tp + fp
    n_discard_gt = tn + fn
    fdr = fp / n_use_gt if n_use_gt > 0 else 0.0
    fur = fn / n_discard_gt if n_discard_gt > 0 else 0.0
    acc = (tp + tn) / total
    return {'FDR': fdr, 'FUR': fur, 'Accuracy': acc, 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn}


metrics = {}
for protocol in ['coldguard', 'mkt', 'vvm']:
    metrics[protocol] = compute_metrics(records, protocol)

df_metrics = pd.DataFrame(metrics).T
print(df_metrics[['FDR', 'FUR', 'Accuracy']].applymap(lambda x: f'{x:.1%}'))

## 4. McNemar's Test (ColdGuard vs MKT)

In [ ]:
def mcnemar_test(records, p1, p2, gt_label=Decision.USE):
    """McNemar's test: does p1 and p2 disagree in a statistically significant way?"""
    b = c = 0  # p1 wrong + p2 right vs p1 right + p2 wrong
    for r in records:
        gt = r['ground_truth']
        p1_correct = (r[p1] == gt)
        p2_correct = (r[p2] == gt)
        if not p1_correct and p2_correct:
            b += 1
        elif p1_correct and not p2_correct:
            c += 1
    chi2_stat = (abs(b - c) - 1) ** 2 / (b + c) if (b + c) > 0 else 0
    p_value = 1 - chi2.cdf(chi2_stat, df=1)
    return {'b': b, 'c': c, 'chi2': chi2_stat, 'p_value': p_value}


print("McNemar ColdGuard vs MKT:", mcnemar_test(records, 'coldguard', 'mkt'))
print("McNemar ColdGuard vs VVM:", mcnemar_test(records, 'coldguard', 'vvm'))

## 5. Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

protocols = ['coldguard', 'mkt', 'vvm']
labels = ['ColdGuard', 'MKT', 'VVM']
colors = ['#2196F3', '#FF9800', '#9C27B0']

fdr_vals = [metrics[p]['FDR'] * 100 for p in protocols]
fur_vals = [metrics[p]['FUR'] * 100 for p in protocols]

x = np.arange(len(protocols))
width = 0.35

axes[0].bar(x - width/2, fdr_vals, width, label='FDR', color='#EF5350')
axes[0].bar(x + width/2, fur_vals, width, label='FUR', color='#42A5F5')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_ylabel('Rate (%)')
axes[0].set_title('False Discard Rate vs False Use Rate')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

acc_vals = [metrics[p]['Accuracy'] * 100 for p in protocols]
axes[1].bar(labels, acc_vals, color=colors)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Overall Decision Accuracy')
axes[1].set_ylim([80, 100])
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../paper/figures/fig_simulation_results.pdf', bbox_inches='tight')
plt.show()
print("Saved to paper/figures/fig_simulation_results.pdf")